In [ ]:
# !pip install spark
# !pip install findspark

In [3]:
import findspark

findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import to_date
from pyspark.sql.types import DoubleType
import pandas as pd

In [6]:
spark = SparkSession.builder.appName("hw6").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/10 06:39:20 WARN Utils: Your hostname, DESKTOP-KV2R5JB, resolves to a loopback address: 127.0.1.1; using 172.22.73.47 instead (on interface eth0)
26/03/10 06:39:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/10 06:39:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [7]:
spark

In [8]:
spark.version

'4.1.1'

In [9]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-10 03:58:53--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.165.49.58, 3.165.49.150, 3.165.49.16, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.165.49.58|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M  25.9MB/s    in 2.6s    

2026-03-10 03:58:56 (25.9 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [2]:
!wget 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'

--2026-03-10 06:38:01--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.165.49.115, 3.165.49.150, 3.165.49.16, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.165.49.115|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-10 06:38:01 (108 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [9]:
path = "./yellow_tripdata_2025-11.parquet"

df = spark.read.parquet(path, header=True, inferSchema=True)
# Repartition the DataFrame to 4 partitions
df_repartitioned = df.repartition(4)

# Save the repartitioned DataFrame to a directory in Parquet format
# The output will be a directory containing 4 parquet files (one for each partition)
output_path = "./parquet_files"
df_repartitioned.write.parquet(output_path, mode="overwrite")

In [10]:
df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [13]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [16]:
df = df.withColumn("tpep_pickup_date_only", to_date("tpep_pickup_datetime"))
df = df.withColumn("tpep_dropoff_date_only", to_date("tpep_dropoff_datetime"))
df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+---------------------+----------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|tpep_pickup_date_only|tpep_dropoff_date_only|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+---------------------+-------------------

In [17]:
df.where("tpep_pickup_date_only = '2025-11-15'").count()

162604

In [31]:
df = df.withColumn("trip_duration", F.col("tpep_dropoff_datetime") - F.col("tpep_pickup_datetime"))
df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+---------------------+----------------------+--------------------+---------------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|tpep_pickup_date_only|tpep_dropoff_date_only|           trip_time|trip_duration_seconds|       trip_duration|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+

In [32]:
df = df.withColumn(
    "trip_duration_hours",
    (F.col("trip_duration").cast("long") / 3600).cast(DoubleType()),
)
df.show(5)

26/03/10 04:38:59 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+---------------------+----------------------+--------------------+---------------------+--------------------+-------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|tpep_pickup_date_only|tpep_dropoff_date_only|           trip_time|trip_duration_seconds|       trip_duration|trip_duration_hours|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+----------

In [35]:
df.agg(F.max("trip_duration_hours")).collect()[0][0]


90.64666666666666

In [23]:
df.createOrReplaceTempView("trips")
df_pulocation = spark.sql("SELECT PULocationID, count(1) FROM trips GROUP BY PULocationID order by count(1)").toPandas()

In [29]:
df_pulocation.head(5)

,PULocationID,count(1)
0,105,1
1,5,1
2,84,1
3,187,3
4,204,4


In [19]:
taxi_zone = spark.read.csv("taxi_zone_lookup.csv", header=True, inferSchema=True).toPandas()
taxi_zone.head()

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [30]:
taxi_zone[taxi_zone.LocationID == 105]

,LocationID,Borough,Zone,service_zone
104,105,Manhattan,Governor's Island/Ellis Island/Liberty Island,Yellow Zone
